In [75]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

### Config

In [77]:
from config import dir_config

raw_dir = Path(dir_config.data.raw)
processed_dir = Path(dir_config.data.processed)

# Following viewing times to be added in reaction time to account for actual accumulation time
min_viewing_time_uw = 0.3
min_viewing_time_ucla = 0.3

### Aggregate Data

### Data is collected from two different set of mice. 
1. UCLA: Big screen, 2 pulse times 
2. UW: smaller screeen, 3 pulse times (check reaction times)

In [78]:
ucla_sessions_trial = list(raw_dir.glob("PSUIM*/*/*_trial.csv"))
ucla_sessions_event = list(raw_dir.glob("PSUIM*/*/*_event.txt"))
ucla_sessions_lick = list(raw_dir.glob("PSUIM*/*/*_lick.txt"))

uw_sessions_trial = list(raw_dir.glob("DBM*/*/*_trial.csv"))
uw_session_licks = list(raw_dir.glob("DBM*/*/*_lick.txt"))

In [79]:
# Initialize empty DataFrames
ucla_data = pd.DataFrame()
uw_data = pd.DataFrame()

# Process UCLA sessions
for session in ucla_sessions_trial:
    temp_df = pd.read_csv(session)
    temp_df["subject_id"] = session.stem.split("_")[0]  # Extract subject_id
    temp_df["session_id"] = session.parent.name  # Get the parent folder name as session_id
    ucla_data = pd.concat([ucla_data, temp_df], ignore_index=True)

# Process UW sessions
for session in uw_sessions_trial:
    temp_df = pd.read_csv(session)
    temp_df["subject_id"] = session.stem.split("_")[0]  # Extract subject_id
    temp_df["session_id"] = session.parent.name  # Get the parent folder name as session_id
    uw_data = pd.concat([uw_data, temp_df], ignore_index=True)

In [80]:
# ----------------------------------------
# PREPARE UCLA DATAFRAME
# ----------------------------------------
ucla_df = pd.DataFrame(
    {
        "data_source": "ucla",
        "subject_id": ucla_data["subject_id"],
        "session_id": ucla_data["session_id"],
        "idx_valid_trial": ucla_data["TrialNo"],
        "idx_correction": ucla_data["AttemptNo"],
        "is_correction_trial": ucla_data["AttemptNo"] != 0,
        "signed_coherence": ucla_data["Coherence"],
        "target": np.where(ucla_data["Correct"] == 1, ucla_data["Response"], -ucla_data["Response"]),
        "choice": ucla_data["Response"],
        "reaction_time": ucla_data["ReactionTime"],
        "outcome": ucla_data["Correct"],
        "reward_amount": ucla_data["RewardAmount"],
        "fixation_duration": ucla_data["FixationTime"],
        "stimulus_duration": ucla_data["StimulusTime"],
        "reinforcement_duration": ucla_data["StimTimeOut"],
        "intertrial_delay": ucla_data["InterTrialInterval"],
        "stimulus_seed": ucla_data["TrialNo"],
    }
)
# Add pulse-related columns
ucla_df["is_pulse_trial"] = ucla_data["BurstCoh"] != 0
ucla_df["pulse_onset"] = ucla_data["BurstTime"]
ucla_df["pulse_duration"] = ucla_data["BurstDur"]
ucla_df["pulse_coherence"] = np.where(ucla_data["BurstCoh"] != 0, np.sign(ucla_data["BurstCoh"] * ucla_df["target"]) * 100, np.nan)  # converting pulse to left/right

In [81]:
# ----------------------------------------
# PREPARE UW DATAFRAME
# ----------------------------------------
uw_df = pd.DataFrame(
    {
        "data_source": "uw",
        "subject_id": uw_data["subject_id"],
        "session_id": uw_data["session_id"],
        "idx_valid_trial": uw_data["idx_valid"],
        "idx_correction": uw_data["idx_correction"],
        "is_correction_trial": uw_data["is_correction_trial"],
        "signed_coherence": uw_data["signed_coherence"],
        "target": uw_data["target"],
        "choice": uw_data["choice"],
        "reaction_time": uw_data["response_time"] + min_viewing_time_uw,  # add min viewing time
        "outcome": uw_data["outcome"],
        "reward_amount": uw_data["trial_reward"],
        "fixation_duration": uw_data["fixation_duration"],
        "stimulus_duration": uw_data["stimulus_duration"],
        "reinforcement_duration": uw_data["reinforcement_duration"],
        "intertrial_delay": uw_data["delay_duration"] + uw_data["intertrial_duration"],
        "is_pulse_trial": uw_data["pulse_coherence"] != uw_data["signed_coherence"],
        "pulse_onset": uw_data["pulse_onset"] * 16.67 / 1000,
        "pulse_duration": uw_data["pulse_duration"] * 16.67 / 1000,
        "pulse_coherence": uw_data["pulse_coherence"],
        "stimulus_seed": uw_data["stimulus_seed"],
    }
)

In [82]:
# ----------------------------------------
# CONCATENATE INTO AGGREGATE DATA
# ----------------------------------------
aggregate_data = pd.concat([ucla_df, uw_df], ignore_index=True)
# insert a column for the index of the session (subject_id, session_id)
aggregate_data.insert(0, "idx_session", pd.Categorical(aggregate_data[["subject_id", "session_id"]].apply(tuple, axis=1)).codes)

In [83]:
# Get number of unique subjects and how many sessions they have
subject_sessions = aggregate_data.groupby("subject_id")["session_id"].nunique().reset_index(name="count")
# remove subjects with less than 2 sessions
subjects_to_remove = subject_sessions[subject_sessions["count"] < 2]["subject_id"]
aggregate_data = aggregate_data[~aggregate_data["subject_id"].isin(subjects_to_remove)]
# Get number of unique subjects and how many sessions they have
subject_sessions = aggregate_data.groupby("subject_id")["session_id"].nunique().reset_index(name="count")
subject_sessions

,subject_id,count
0,DBM11,19
1,DBM12,17
2,DBM13,9
3,DBM14,20
4,DBM15,50
5,DBM16,14
6,DBM5,34
7,DBM6,33
8,DBM8,22
9,PSUIM10,9


In [84]:
ucla_data.to_csv(Path(raw_dir, "ucla_data.csv"), index=False)
uw_data.to_csv(Path(raw_dir, "uw_data.csv"), index=False)

aggregate_data.to_csv(Path(raw_dir, "aggregate_data.csv"), index=False)